# BirdCLEF 2026 Inference (Single-Model, Self-Contained)

This notebook runs Kaggle inference using the best checkpoint from our NFNet setup.
It does not import any project files. Edit the config paths below to point to your model.


In [ ]:
from pathlib import Path
from typing import Dict, Iterable, List, Tuple
import json
import os

import pandas as pd
import torch
import torch.nn.functional as F
import torchaudio
import torchaudio.functional as AF
import soundfile as sf
from tqdm import tqdm

try:
    import timm
except Exception as e:
    raise RuntimeError('timm is required for this notebook') from e

# =========================
# Config (edit these)
# =========================
COMP_INPUT = Path('/kaggle/input/birdclef-2026')
SOUNDSCAPE_DIR = COMP_INPUT / 'test_soundscapes'
# Prefer test.csv if present, otherwise try sample_submission.csv
METADATA_CANDIDATES = [COMP_INPUT / 'test.csv', COMP_INPUT / 'sample_submission.csv']
METADATA_CSV = next((p for p in METADATA_CANDIDATES if p.exists()), None)

# Update this path to your uploaded model checkpoint
MODEL_PATH = Path('/kaggle/input/your-model/best_model.pt')
LABEL_MAP_PATH = MODEL_PATH.parent / 'label_map.json'  # optional
OUTPUT_CSV = Path('submission.csv')

SAMPLE_RATE = 32000
SEGMENT_DURATION = 5.0
HOP_DURATION = 5.0
BATCH_SIZE = 32
PRELOAD_WORKERS = 4

print('SOUNDSCAPE_DIR:', SOUNDSCAPE_DIR, SOUNDSCAPE_DIR.exists())
print('METADATA_CSV:', METADATA_CSV)
print('MODEL_PATH:', MODEL_PATH, MODEL_PATH.exists())
print('LABEL_MAP_PATH:', LABEL_MAP_PATH, LABEL_MAP_PATH.exists())


In [ ]:
def _read_audio(filepath: Path) -> Tuple[torch.Tensor, int]:
    data, sr = sf.read(str(filepath), dtype='float32')
    if data.ndim == 1:
        waveform = torch.from_numpy(data).unsqueeze(0)
    else:
        waveform = torch.from_numpy(data.T.copy())
    return waveform, sr


class SoundscapeSampler:
    def __init__(
        self,
        sample_rate: int = 32000,
        duration: float = 5.0,
        hop: float = 5.0,
        preload_audio: bool = True,
        max_cached_files: int = 100,
        preload_workers: int = 1,
    ):
        if hop <= 0:
            raise ValueError('hop must be positive')
        self.sample_rate = sample_rate
        self.duration = duration
        self.segment_samples = int(duration * sample_rate)
        self.hop_samples = int(hop * sample_rate)
        self.preload_audio = preload_audio
        self.max_cached_files = max_cached_files
        self.preload_workers = max(1, preload_workers)
        self._audio_cache = {}

    def _load_audio_file(self, soundscape_path: Path) -> torch.Tensor:
        waveform, sr = _read_audio(soundscape_path)
        if waveform.size(0) > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if sr != self.sample_rate:
            waveform = AF.resample(waveform, orig_freq=sr, new_freq=self.sample_rate)
        return waveform.contiguous()

    def _load_cached_audio(self, soundscape_path: Path) -> torch.Tensor:
        cached = self._audio_cache.get(soundscape_path)
        if cached is not None:
            return cached
        waveform = self._load_audio_file(soundscape_path)
        self._audio_cache[soundscape_path] = waveform
        if not self.preload_audio and len(self._audio_cache) > self.max_cached_files:
            self._audio_cache.pop(next(iter(self._audio_cache)))
        return waveform

    def iterate_segment_items(self, soundscape_path: Path):
        waveform = self._load_cached_audio(soundscape_path)
        total = waveform.size(1)
        if total == 0:
            return
        start = 0
        while start < total:
            end = start + self.segment_samples
            chunk = waveform[:, start:end]
            if chunk.size(1) < self.segment_samples:
                pad = self.segment_samples - chunk.size(1)
                chunk = F.pad(chunk, (0, pad))
            end_seconds = int(end / self.sample_rate)
            yield end_seconds, chunk
            start += self.hop_samples


class SpectrogramTransform(torch.nn.Module):
    def __init__(
        self,
        sample_rate: int = 32000,
        n_mels: int = 160,
        n_fft: int = 2048,
        hop_length: int = 512,
        f_min: int = 20,
        f_max: int | None = None,
        normalize: bool = True,
    ):
        super().__init__()
        self.melspec = torchaudio.transforms.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=n_fft,
            hop_length=hop_length,
            n_mels=n_mels,
            f_min=f_min,
            f_max=f_max or sample_rate // 2,
            power=2.0,
        )
        self.to_db = torchaudio.transforms.AmplitudeToDB(stype='power')
        self.normalize = normalize

    def forward(self, waveform: torch.Tensor) -> torch.Tensor:
        spec = self.melspec(waveform)
        spec = self.to_db(spec)
        if self.normalize:
            mean = spec.mean(dim=[-2, -1], keepdim=True)
            std = spec.std(dim=[-2, -1], keepdim=True).clamp(min=1e-6)
            spec = (spec - mean) / std
        return spec


class TimmSpectrogramClassifier(torch.nn.Module):
    def __init__(
        self,
        n_classes: int,
        backbone_name: str,
        image_size: int,
        dropout: float,
        sample_rate: int,
        n_mels: int,
        n_fft: int,
        hop_length: int,
        freq_mask_param: int = 0,
        time_mask_param: int = 0,
        specaugment_masks: int = 0,
        spec_noise_std: float = 0.0,
    ):
        super().__init__()
        self.image_size = int(image_size)
        self.spectrogram = SpectrogramTransform(
            sample_rate=sample_rate,
            n_mels=n_mels,
            n_fft=n_fft,
            hop_length=hop_length,
        )
        self.backbone = timm.create_model(
            backbone_name,
            pretrained=False,
            in_chans=3,
            num_classes=n_classes,
            drop_rate=dropout,
        )

    def forward(self, waveform: torch.Tensor) -> torch.Tensor:
        x = self.spectrogram(waveform)
        x = x.clamp(min=-4.0, max=4.0)
        x = (x + 4.0) / 8.0
        x = F.interpolate(x, size=(self.image_size, self.image_size), mode='bilinear', align_corners=False)
        x = x.repeat(1, 3, 1, 1)
        return self.backbone(x)


def load_label_map(checkpoint: Dict, label_map_path: Path | None) -> Dict[str, int]:
    if label_map_path and label_map_path.exists():
        return json.loads(label_map_path.read_text())
    return checkpoint.get('label_map', {})


def iterate_batches(segments, batch_size: int):
    time_buffer = []
    waveform_buffer = []
    for segment_end_seconds, waveform in segments:
        time_buffer.append(segment_end_seconds)
        waveform_buffer.append(waveform)
        if len(waveform_buffer) == batch_size:
            yield time_buffer, torch.stack(waveform_buffer)
            time_buffer, waveform_buffer = [], []
    if waveform_buffer:
        yield time_buffer, torch.stack(waveform_buffer)


In [ ]:
if METADATA_CSV is None:
    raise FileNotFoundError('Could not locate test metadata CSV in competition input')

checkpoint = torch.load(MODEL_PATH, map_location='cpu')
label_map = load_label_map(checkpoint, LABEL_MAP_PATH if LABEL_MAP_PATH.exists() else None)
if not label_map:
    raise ValueError('label map is required either via label_map.json or checkpoint')
reverse_labels = sorted(label_map.keys(), key=lambda label: label_map[label])

model_config = checkpoint.get('model_config', {})
if not model_config:
    raise ValueError('model_config missing in checkpoint')

model = TimmSpectrogramClassifier(
    n_classes=len(label_map),
    backbone_name=model_config.get('backbone_name', 'eca_nfnet_l0'),
    image_size=int(model_config.get('image_size', 288)),
    dropout=float(model_config.get('dropout', 0.35)),
    sample_rate=int(model_config.get('sample_rate', SAMPLE_RATE)),
    n_mels=int(model_config.get('n_mels', 160)),
    n_fft=int(model_config.get('n_fft', 2048)),
    hop_length=int(model_config.get('hop_length', 512)),
)
model.load_state_dict(checkpoint['model_state'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

sampler = SoundscapeSampler(
    sample_rate=SAMPLE_RATE,
    duration=SEGMENT_DURATION,
    hop=HOP_DURATION,
    preload_audio=True,
    preload_workers=PRELOAD_WORKERS,
)

metadata = pd.read_csv(METADATA_CSV)
if 'filename' in metadata.columns:
    filenames = metadata['filename'].dropna().astype(str).drop_duplicates().tolist()
elif 'row_id' in metadata.columns:
    filenames = (
        metadata['row_id']
        .dropna()
        .astype(str)
        .str.replace(r'_[0-9]+$', '', regex=True)
        .drop_duplicates()
        .map(lambda stem: f'{stem}.ogg')
        .tolist()
    )
else:
    raise ValueError('Metadata must contain filename or row_id column')

predictions = []
for filename in tqdm(filenames, desc='Predict'):
    filepath = SOUNDSCAPE_DIR / filename
    if not filepath.exists():
        raise FileNotFoundError(f'Missing soundscape {filepath}')

    for end_seconds, batch in iterate_batches(
        sampler.iterate_segment_items(filepath),
        batch_size=BATCH_SIZE,
    ):
        batch = batch.to(device, non_blocking=True)
        with torch.no_grad():
            probs = torch.sigmoid(model(batch)).cpu()
        for idx, prob in enumerate(probs):
            row_dict = {'row_id': f'{Path(filename).stem}_{end_seconds[idx]}'}
            row_dict.update({label: float(prob[j].item()) for j, label in enumerate(reverse_labels)})
            predictions.append(row_dict)

if not predictions:
    raise RuntimeError('No soundscapes were scored; check input metadata and files.')

submission = pd.DataFrame(predictions)
if 'row_id' in metadata.columns:
    submission = metadata[['row_id']].merge(submission, on='row_id', how='left')
submission = submission[['row_id'] + reverse_labels]
submission.to_csv(OUTPUT_CSV, index=False)
print(f'Saved predictions to {OUTPUT_CSV}')
